In [ ]:
# Обучение модели на Т4-2 шт.
# -*- coding: utf-8 -*-
# ИТОГОВЫЙ СКРИПТ ОБУЧЕНИЯ ДЛЯ BIRDCLEF+ 2026
# Сочетает лучшее из: ResNet50 + Focal Loss + SpecAugment + K-Fold
# Полная поддержка чекпоинтов (каждая эпоха каждого фолда)
# ИСПРАВЛЕННАЯ ВЕРСИЯ — правильный парсинг имён чекпоинтов

import os
import glob
import json
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torchaudio
import librosa
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from tqdm import tqdm
from sklearn.model_selection import KFold
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# ============ НАСТРОЙКИ ============
DATA_PATH = '/kaggle/input/birdclef-2026'
BATCH_SIZE = 48
EPOCHS = 15
LEARNING_RATE = 0.001
NUM_WORKERS = 4
N_FOLDS = 3
EARLY_STOPPING_PATIENCE = 3
MIXUP_ALPHA = 0.4          # Из топ-скрипта
FOCAL_GAMMA = 2.5          # Из топ-скрипта
LABEL_SMOOTHING = 0.03     # Из топ-скрипта

# Директории для чекпоинтов
CHECKPOINT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Файл для отслеживания прогресса
PROGRESS_FILE = '/kaggle/working/training_progress.json'

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Используется устройство: {DEVICE}")
print(f"📦 Количество GPU: {torch.cuda.device_count()}")
print(f"📦 Размер батча: {BATCH_SIZE}")
print(f"🔄 Количество фолдов: {N_FOLDS}")

# ============ ЗАГРУЗКА ДАННЫХ ============
train_df = pd.read_csv(os.path.join(DATA_PATH, 'train.csv'))
taxonomy = pd.read_csv(os.path.join(DATA_PATH, 'taxonomy.csv'))
species_list = taxonomy['primary_label'].tolist()
species_to_idx = {species: idx for idx, species in enumerate(species_list)}
print(f"✅ Загружено {len(species_list)} видов")

# ============ ФУНКЦИИ ОБРАБОТКИ АУДИО ============
def load_audio(file_path, target_sr=32000, duration=5.0):
    try:
        waveform, sr = torchaudio.load(file_path)
        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)
        if sr != target_sr:
            resampler = torchaudio.transforms.Resample(sr, target_sr)
            waveform = resampler(waveform)
        target_length = int(target_sr * duration)
        if waveform.shape[1] < target_length:
            padding = target_length - waveform.shape[1]
            waveform = torch.nn.functional.pad(waveform, (0, padding))
        else:
            waveform = waveform[:, :target_length]
        return waveform.squeeze().numpy().astype(np.float32), target_sr
    except:
        return np.zeros(int(target_sr * duration), dtype=np.float32), target_sr

def create_mel_spectrogram(audio, sr=32000, n_mels=128):
    hop_length = 512
    win_length = 2048
    mel_spec = librosa.feature.melspectrogram(
        y=audio, sr=sr, n_mels=n_mels,
        hop_length=hop_length, win_length=win_length
    )
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    mel_spec_db = (mel_spec_db - mel_spec_db.mean()) / (mel_spec_db.std() + 1e-8)
    return mel_spec_db.astype(np.float32)

# ============ УЛУЧШЕННАЯ АУГМЕНТАЦИЯ ============
class SpecAugment:
    def __init__(self, time_mask=30, freq_mask=15):
        self.time_mask = time_mask
        self.freq_mask = freq_mask

    def __call__(self, mel_spec):
        mel_spec = mel_spec.copy()
        # Time masking
        if np.random.random() > 0.5:
            t = np.random.randint(0, self.time_mask)
            t0 = np.random.randint(0, max(1, mel_spec.shape[1] - t))
            mel_spec[:, t0:t0+t] = 0
        # Frequency masking
        if np.random.random() > 0.5:
            f = np.random.randint(0, self.freq_mask)
            f0 = np.random.randint(0, max(1, mel_spec.shape[0] - f))
            mel_spec[f0:f0+f, :] = 0
        # Добавляем небольшой шум
        if np.random.random() > 0.5:
            noise = np.random.normal(0, 0.005, mel_spec.shape)
            mel_spec = mel_spec + noise
        return mel_spec

def mixup(spec1, spec2, alpha=MIXUP_ALPHA):
    lam = np.random.beta(alpha, alpha)
    mixed_spec = lam * spec1 + (1 - lam) * spec2
    return mixed_spec, lam

spec_augment = SpecAugment()

# ============ LABEL SMOOTHING LOSS ============
class LabelSmoothingBCEWithLogitsLoss(nn.Module):
    def __init__(self, smoothing=LABEL_SMOOTHING):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, pred, target):
        target = target * (1 - self.smoothing) + self.smoothing / target.size(1)
        return nn.functional.binary_cross_entropy_with_logits(pred, target)

# ============ FOCAL LOSS ============
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=FOCAL_GAMMA):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        bce_loss = nn.functional.binary_cross_entropy_with_logits(inputs, targets, reduction='none')
        pt = torch.exp(-bce_loss)
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

# ============ СМЕШАННАЯ ФУНКЦИЯ ПОТЕРЬ ============
class CombinedLoss(nn.Module):
    def __init__(self, focal_weight=0.7, bce_weight=0.3, smoothing=LABEL_SMOOTHING):
        super().__init__()
        self.focal = FocalLoss()
        self.bce = LabelSmoothingBCEWithLogitsLoss(smoothing)
        self.focal_weight = focal_weight
        self.bce_weight = bce_weight

    def forward(self, pred, target):
        return self.focal_weight * self.focal(pred, target) + self.bce_weight * self.bce(pred, target)

# ============ DATASET КЛАСС ============
class BirdCLEFDataset(Dataset):
    def __init__(self, data_list, species_to_idx, is_train=True):
        self.data_list = data_list
        self.species_to_idx = species_to_idx
        self.is_train = is_train

    def __len__(self):
        return len(self.data_list)

    def __getitem__(self, idx):
        row, audio_path = self.data_list[idx]
        audio, sr = load_audio(audio_path)
        mel_spec = create_mel_spectrogram(audio, sr)

        if self.is_train and np.random.random() > 0.6:
            mel_spec = spec_augment(mel_spec)

        mel_spec = torch.FloatTensor(mel_spec).unsqueeze(0)
        mel_spec = torch.nn.functional.interpolate(
            mel_spec.unsqueeze(0), size=(224, 224), mode='bilinear'
        ).squeeze(0)

        if self.is_train:
            label = torch.zeros(len(self.species_to_idx))
            label[self.species_to_idx[row['primary_label']]] = 1.0
            return mel_spec, label
        return mel_spec

# ============ МОДЕЛЬ (ResNet50) ============
class BirdCLEFModel(nn.Module):
    def __init__(self, num_classes=234):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
        self.backbone.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Linear(in_features, num_classes)

    def forward(self, x):
        return self.backbone(x)

# ============ ФУНКЦИИ ДЛЯ ЧЕКПОИНТОВ (ИСПРАВЛЕННЫЕ) ============
def save_checkpoint(epoch, fold, model, optimizer, scheduler, scaler, train_loss, val_loss, is_best=False):
    checkpoint = {
        'epoch': epoch,
        'fold': fold,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
    }
    checkpoint_path = os.path.join(CHECKPOINT_DIR, f'checkpoint_fold{fold}_epoch{epoch}.pth')
    torch.save(checkpoint, checkpoint_path)

    if is_best:
        best_path = '/kaggle/working/birdclef_best_model.pth'
        model_to_save = model.module if hasattr(model, 'module') else model
        torch.save(model_to_save.state_dict(), best_path)
        print(f"💾 Сохранена лучшая модель (val_loss: {val_loss:.4f})")
    return checkpoint_path

def get_last_checkpoint_info():
    """Находит последний чекпоинт и возвращает информацию о нём"""
    checkpoints = glob.glob(os.path.join(CHECKPOINT_DIR, 'checkpoint_fold*_epoch*.pth'))
    if not checkpoints:
        return None, None, None

    latest = None
    latest_fold = -1
    latest_epoch = -1

    for cp in checkpoints:
        basename = os.path.basename(cp)
        parts = basename.replace('.pth', '').split('_')

        # parts[0] = "checkpoint"
        # parts[1] = "fold1" -> нужно "1"
        # parts[2] = "epoch5" -> нужно "5"
        fold_str = parts[1].replace('fold', '')
        epoch_str = parts[2].replace('epoch', '')

        try:
            fold = int(fold_str)
            epoch = int(epoch_str)
        except ValueError:
            print(f"⚠️ Не удалось распарсить имя файла: {basename}")
            continue

        if fold > latest_fold or (fold == latest_fold and epoch > latest_epoch):
            latest = cp
            latest_fold = fold
            latest_epoch = epoch

    return latest, latest_fold, latest_epoch

def load_checkpoint(model, optimizer, scheduler, scaler, checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    scaler.load_state_dict(checkpoint['scaler_state_dict'])
    return checkpoint['epoch'], checkpoint['fold'], checkpoint['val_loss']

def save_progress_state(fold, completed_folds, best_val_losses):
    state = {
        'current_fold': fold,
        'completed_folds': completed_folds,
        'best_val_losses': best_val_losses,
    }
    with open(PROGRESS_FILE, 'w') as f:
        json.dump(state, f)

def load_progress_state():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, 'r') as f:
            return json.load(f)
    return None

# ============ ПОДГОТОВКА ДАННЫХ ============
print("\n📊 Подготовка данных...")
valid_data = []
for idx, row in train_df.iterrows():
    audio_path = os.path.join(DATA_PATH, 'train_audio', row['filename'])
    if os.path.exists(audio_path):
        valid_data.append((row, audio_path))
    if len(valid_data) >= 30000:
        break

print(f"✅ Найдено {len(valid_data)} валидных аудиофайлов")

# Подготовка для стратификации
labels_list = [row[0]['primary_label'] for row in valid_data]
unique_labels = list(set(labels_list))
label_to_int = {lbl: i for i, lbl in enumerate(unique_labels)}
y = [label_to_int[lbl] for lbl in labels_list]

# K-Fold
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# ============ ЗАГРУЗКА ПРОГРЕССА ============
progress = load_progress_state()
start_fold = 0
completed_folds = []
best_val_losses = [float('inf')] * N_FOLDS

if progress:
    start_fold = progress.get('current_fold', 0)
    completed_folds = progress.get('completed_folds', [])
    best_val_losses = progress.get('best_val_losses', [float('inf')] * N_FOLDS)
    print(f"\n🔄 Загружен прогресс: завершены фолды {completed_folds}")
else:
    print("\n🚀 Начинаем обучение с нуля")

# ============ ОБУЧЕНИЕ ============
for fold, (train_idx, val_idx) in enumerate(kf.split(valid_data, y)):

    # Пропускаем уже завершённые фолды
    if fold in completed_folds:
        print(f"\n{'='*60}")
        print(f"⏩ Фолд {fold+1}/{N_FOLDS} уже завершён, пропускаем")
        print(f"🏆 Лучшая val_loss: {best_val_losses[fold]:.4f}")
        print(f"{'='*60}")
        continue

    # Пропускаем фолды до того, на котором остановились
    if fold < start_fold:
        continue

    print(f"\n{'='*60}")
    print(f"🚀 Фолд {fold+1}/{N_FOLDS}")
    print(f"{'='*60}")

    train_data = [valid_data[i] for i in train_idx]
    val_data = [valid_data[i] for i in val_idx]

    train_dataset = BirdCLEFDataset(train_data, species_to_idx, is_train=True)
    val_dataset = BirdCLEFDataset(val_data, species_to_idx, is_train=True)

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = BirdCLEFModel(num_classes=len(species_list))

    if torch.cuda.device_count() > 1:
        print(f"🚀 Используем {torch.cuda.device_count()} GPU")
        model = nn.DataParallel(model)

    model = model.to(DEVICE)

    criterion = CombinedLoss(focal_weight=0.7, bce_weight=0.3, smoothing=LABEL_SMOOTHING)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5, T_mult=2)
    scaler = torch.cuda.amp.GradScaler()

    # Загрузка чекпоинта
    start_epoch = 0
    best_val_loss = float('inf')
    last_checkpoint, last_fold, last_epoch = get_last_checkpoint_info()

    if last_fold == fold + 1 and last_epoch is not None:
        print(f"🔄 Найден чекпоинт для фолда {fold+1}, эпоха {last_epoch}")
        try:
            loaded_epoch, loaded_fold, loaded_loss = load_checkpoint(
                model, optimizer, scheduler, scaler, last_checkpoint
            )
            start_epoch = loaded_epoch
            best_val_loss = loaded_loss
            print(f"✅ Загружен чекпоинт: эпоха {start_epoch}, val_loss: {best_val_loss:.4f}")
        except Exception as e:
            print(f"⚠️ Ошибка загрузки чекпоинта: {e}, начинаем с нуля")
            start_epoch = 0
    else:
        print(f"🚀 Начинаем фолд {fold+1} с нуля")

    patience_counter = 0

    for epoch in range(start_epoch, EPOCHS):
        # Training
        model.train()
        train_loss = 0
        loop = tqdm(train_loader, desc=f'Epoch {epoch+1}')
        for inputs, labels in loop:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

            # MixUp аугментация
            if np.random.random() > 0.5:
                batch_size = inputs.size(0)
                indices = torch.randperm(batch_size).to(DEVICE)
                lam = np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA)
                inputs = lam * inputs + (1 - lam) * inputs[indices]
                labels = lam * labels + (1 - lam) * labels[indices]

            optimizer.zero_grad()
            with torch.cuda.amp.autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            train_loss += loss.item()
            loop.set_postfix(loss=loss.item())

        # Validation
        model.eval()
        val_loss = 0
        with torch.no_grad():
            for inputs, labels in tqdm(val_loader, desc='Validating'):
                inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
                with torch.cuda.amp.autocast():
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                val_loss += loss.item()

        train_loss /= len(train_loader)
        val_loss /= len(val_loader)

        # Обновляем scheduler
        if isinstance(scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau):
            scheduler.step(val_loss)
        else:
            scheduler.step()

        current_lr = optimizer.param_groups[0]['lr']
        print(f"📊 Фолд {fold+1} | Эпоха {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | LR: {current_lr:.6f}")

        is_best = val_loss < best_val_loss
        if is_best:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1

        # Сохраняем чекпоинт КАЖДОЙ ЭПОХИ
        save_checkpoint(epoch + 1, fold + 1, model, optimizer, scheduler, scaler, train_loss, val_loss, is_best)
        save_progress_state(fold, completed_folds, best_val_losses)

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"⏹️ Early stopping на эпохе {epoch+1} для фолда {fold+1}")
            break

    # Фолд завершён
    completed_folds.append(fold)
    best_val_losses[fold] = best_val_loss
    save_progress_state(fold + 1, completed_folds, best_val_losses)
    print(f"🏆 Лучшая val_loss фолда {fold+1}: {best_val_loss:.4f}")

print("\n" + "="*60)
print("✅ ОБУЧЕНИЕ ЗАВЕРШЕНО!")
print("="*60)
print(f"📊 Результаты по фолдам:")
for i, loss in enumerate(best_val_losses):
    print(f"   Фолд {i+1}: {loss:.4f}")
print(f"💾 Лучшая модель сохранена в /kaggle/working/birdclef_best_model.pth")
print(f"💾 Все чекпоинты сохранены в {CHECKPOINT_DIR}")

from IPython.display import FileLink
print(f"\n📥 Скачать лучшую модель: {FileLink('/kaggle/working/birdclef_best_model.pth')}")
